

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [82]:
# Read required tokens
from google.colab import userdata
hf_api_key = userdata.get("HF_TOKEN")
openai_api_key = userdata.get("OPENAI_TOKEN")
serpapi_api_key = userdata.get("SERPAPI_TOKEN")

In [86]:
# Set constants
temperature_moderate = 0.5
temperature_conservative = 0.1

In [87]:
# Installs
!pip install -q -U mypy-extensions # neede for current version of LangChain

In [88]:
!pip -q install langchain langchain_openai langchain-community langchain_experimental langchain_classic

In [89]:
!pip install -q google-search-results duckduckgo_search

In [91]:
!pip install -U -q ddgs

In [93]:
!pip show langchain langchain-core langchain-community langchain-experimental langchain-classic langchainhub 2>/dev/null | grep -E "^(Name|Version)"

Name: langchain
Version: 1.3.4
Name: langchain-core
Version: 1.4.0
Name: langchain-community
Version: 0.4.2
Name: langchain-experimental
Version: 0.4.2
Name: langchain-classic
Version: 1.0.7


In [94]:
# Imports
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import os # to set environmental variable

Параметр temperature у наступній моделі буде встановлено на 0.1 (temperature_conservative) так як ми хочемо генерувати текст найбільш близький до інформації з джерел.

In [95]:
# Create model instance
llm_openai = ChatOpenAI(temperature=temperature_conservative, api_key=openai_api_key)
llm_openai

ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7bd8d00e1940>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7bd8d00e1460>, root_client=<openai.OpenAI object at 0x7bd8d00e3e60>, root_async_client=<openai.AsyncOpenAI object at 0x7bd8d00e3da0>, temperature=0.1, model_kwargs={}, openai_api_key=SecretStr('**********')

In [96]:
# Create prompt: limit answer to 200 words since 200 characters limit is too small (200 characters output is given below in text)
prompt_quant_msg = [
    ("system", """You are a technical expert.
    Provide a concise overview on the subject using simple and clear language.
    Your answer should contain: key definitions in the domain, key advantages of the subject, current research topics on the subject.
    Important: the entire response must be under 200 words total."""),
    ("human", "What are quantum calculations?")
]

In [97]:
response = llm_openai.invoke(prompt_quant_msg)
print(response.content)

Quantum calculations involve using principles of quantum mechanics to perform computations. Quantum bits (qubits) can exist in multiple states simultaneously, allowing for parallel processing and potentially faster calculations compared to classical computers. 

Key advantages of quantum calculations include the ability to solve complex problems more efficiently, such as cryptography, optimization, and simulation tasks. 

Current research topics in quantum calculations include developing error correction techniques to improve the reliability of quantum computers, exploring quantum machine learning algorithms, and investigating the potential impact of quantum computing on various industries.


Відповідь моделі, коли поставлено ліміт 200 знаків:

Quantum calculations use quantum mechanics to process information.
They offer advantages in speed and efficiency.
Current research focuses on error correction and scaling algorithms.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [98]:
# Imports
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [99]:
# Create prompt: limit answer to 200 words since 200 characters limit is too small (200 characters output is given below in text)
prompt_templ_explain = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        """You are a technical expert.
        Provide a concise overview on the subject using simple and clear language.
        Your answer should contain: key definitions in the domain, key advantages of the subject, current research topics on the subject.
        Important: the entire response must be under 200 words total."""
    ),
    HumanMessagePromptTemplate.from_template("{user_question}")
])

In [100]:
# Create chain
chain = prompt_templ_explain | llm_openai

In [125]:
# Derive the responses from the model
response = chain.invoke({"user_question": "What are Bayesian methods in Machine Learning?"})
print(response.content)

Bayesian methods in Machine Learning use Bayes' theorem to update the probability of a hypothesis as more evidence or data becomes available. Key advantages include handling uncertainty, incorporating prior knowledge, and providing a principled way to make predictions. Current research topics include Bayesian deep learning, probabilistic programming, and Bayesian optimization.


In [103]:
response = chain.invoke({"user_question": "What are Transformers in Machine Learning?"})
print(response.content)

Transformers are a type of deep learning model that has gained popularity for various natural language processing tasks. They use self-attention mechanisms to weigh the importance of different words in a sentence when making predictions. This allows them to capture long-range dependencies in data more effectively compared to traditional models.

Key advantages of transformers include their ability to handle sequential data efficiently, their parallelizability, and their capability to learn complex patterns in data without relying on predefined features.

Current research topics on transformers include improving their efficiency for training on large datasets, developing more interpretable models, and adapting them to different domains such as computer vision and reinforcement learning.


In [104]:
response = chain.invoke({"user_question": "What is Explainable AI?"})
print(response.content)

Explainable AI (XAI) refers to the development of AI systems that can provide understandable explanations of their decisions and actions to humans. Key advantages include increased transparency, accountability, and trust in AI systems. Current research topics in XAI focus on creating interpretable models, generating explanations for complex AI algorithms, and designing user-friendly interfaces for presenting explanations to non-experts.




### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [126]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

"Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. Obama’s father, also named Barack Hussein Obama, grew up in a small village in Nyanza Province, Kenya, as a member of the Luo ethnicity. Obama was elected the first African American editor of the Harvard Law... Valerie Jarrett Self - Obama Special Advisor (2 episodes, 2008-2014). Obama's first name trivia, fun quiz questions, history quiz online, guess the president's name, presidential trivia questions, quiz about Obama, educational quizzes for adults..."

In [127]:
# Imports
from langchain_classic.agents import load_tools, Tool, AgentExecutor, AgentType, create_react_agent, initialize_agent
from langchain_core.prompts import PromptTemplate

In [128]:
# Create a prompt for a React agent
template = """You are an expert scientific research assistant. Your goal is to provide accurate, evidence-based answers by executing a systematic literature search.

TARGET DATABASES: You must prioritize sources from SpringerNature Link, IEEE Xplore, ScienceDirect, ACM Library, Nature.

CRITICAL RESEARCH STRATEGY:
1. First, search in given target databases to identify 5 MOST RECENT distinct scientific papers on the topic.
2. Prioritize scientific papers indexed with a DOI when available in the search results.

You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the scientific search tool (use precise academic keywords or paper titles)
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)

Thought: I have gathered specific details (titles, authors, years) for 5 recent papers and am ready to write the Final Answer.

Final Answer:
(You MUST list 5 most recent specific papers you found in your search observations. For EACH paper, you must provide this exact structure.)
- **[Year]**:[Publication Year]
- *Authors*: [Author Names or "Unknown" if missing from snippet]
- *Title*: [Exact Publication Title]
- *Summary*: [2-3 sentence summary of key findings based ONLY on your search observations]
- *Published in/Publisher*: [Name of the Journal, Conference, or Publishing Institution if visible; otherwise list the source domain]
- *URL*: [Web-link if available in the search metadata, otherwise omit]

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(template=template, input_variables=["tools", "tool_names", "input", "agent_scratchpad"])

In [113]:
# Loading SerpAPI
search_tool = load_tools(["serpapi"], llm=llm_openai, serpapi_api_key=serpapi_api_key)

In [114]:
# Agent
scientist_agent = create_react_agent(llm=llm_openai, tools=search_tool, prompt=prompt)

agent_executor = AgentExecutor(agent=scientist_agent, tools=search_tool, verbose=True, max_iterations=10, handle_parsing_errors=True) # max_iterations=10 - to look for multiple sources if needed

In [117]:
# Run the agent
agent_executor.invoke({'input': "What are the most recent scientific papers in Artificial Intelligence?"})



> Entering new AgentExecutor chain...
I need to search for the 5 most recent distinct scientific papers on Artificial Intelligence in the target databases.
Action: Search
Action Input: "Artificial Intelligence recent research papers"['Artificial Intelligence. Authors and titles for recent submissions. Fri, 5 Jun 2026 · Thu, 4 Jun 2026 · Wed, 3 Jun 2026 · Tue, 2 Jun 2026 · Mon, 1 Jun 2026.', 'Articles · Differential Parity: Relative Fairness Between Two Sets of Decisions · R-Mod: Minimal Structural Revision of S5 Epistemic Models · Deep Learning Agents ...', 'Keep up with the latest in AI research. Follow the latest in generative AI research papers and stay ahead of cutting-edge advancements.', 'Will AI do the same? A new study of the postwar U.S. shows which kinds of workers historically filled new tech-enabled jobs. May 21, 2026. Read full story ...', 'Today, I return to TDS with my AI papers to read series. My long-term followers might recall the four previous editions ([1], [2], [

{'input': 'What are the most recent scientific papers in Artificial Intelligence?',
 'output': "- **[2026]**:\n- *Authors*: Unknown\n- *Title*: Differential Parity: Relative Fairness Between Two Sets of Decisions\n- *Summary*: This paper explores the concept of relative fairness between two sets of decisions in the context of AI.\n- *Published in/Publisher*: Springer Nature\n\n- **[2026]**:\n- *Authors*: Unknown\n- *Title*: R-Mod: Minimal Structural Revision of S5 Epistemic Models\n- *Summary*: The paper discusses the minimal structural revision of S5 Epistemic Models using AI.\n- *Published in/Publisher*: Springer Nature\n\n- **[2026]**:\n- *Authors*: Unknown\n- *Title*: Will AI do the same? A new study of the postwar U.S. shows which kinds of workers historically filled new tech-enabled jobs\n- *Summary*: This study examines the historical trends of workers filling tech-enabled jobs in the postwar U.S.\n- *Published in/Publisher*: Springer Nature\n\n- **[2026]**:\n- *Authors*: Unknow



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [129]:
# Imports
from langchain_experimental.utilities import PythonREPL # for Python interpreter

In [157]:
# Define LLM
llm_task4 = ChatOpenAI(temperature=temperature_moderate, api_key=openai_api_key) # temperature_moderate - to allow more freedom in answering questions

In [158]:
# Define tool for operations with Python
python_repl = PythonREPL()
python_tool = Tool(name="python_interpreter", func=python_repl.run, description="Use for any complex calculations, loops, or logic. Input must be valid Python code that uses print() to output the result.")

In [159]:
# Define search tool
tools_task4 = load_tools(["serpapi"], llm=llm_task4, serpapi_api_key=serpapi_api_key) # loading SerpAPI

In [160]:
# List of all tools
tools_task4.append(python_tool)
tools_task4

[Tool(name='Search', description='A search engine. Useful for when you need to answer questions about current events. Input should be a search query.', func=<bound method SerpAPIWrapper.run of SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'google_domain': 'google.com', 'gl': 'us', 'hl': 'en'}, serpapi_api_key='0097b874568994a89490372e8abe33cf94a2de1eac83f629a9624a559d79872f', aiosession=None)>, coroutine=<bound method SerpAPIWrapper.arun of SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'google_domain': 'google.com', 'gl': 'us', 'hl': 'en'}, serpapi_api_key='0097b874568994a89490372e8abe33cf94a2de1eac83f629a9624a559d79872f', aiosession=None)>),
 Tool(name='python_interpreter', description='Use for any complex calculations, loops, or logic. Input must be valid Python code that uses print() to output the result.', func=<bound method PythonREPL.run of PythonREPL(globals={}, loca

In [161]:
# Create a prompt for a React agent
template_task4 = """You are a business expert in the field of international trade, agriculture, and market dynamics specializing in the citrus industry.

Your goal is to provide an accurate export forecast for the upcoming year by executing a comparative analysis:
1. Identify the baseline trends using the historical data (years and volumes) provided directly in the user's question.
2. Investigate the historical weather conditions and economic/inflation factors that corresponded to those baseline years.
3. Search for the upcoming/forecasted weather and economic conditions for the target forecasting year.
4. Synthesize your findings: Adjust the historical baseline up or down based on how the upcoming year's weather and economic situation compare to the past.

CRITICAL: If you lack the data needed to provide a reliable forecast, or if the tools do not provide definitive information, explicitly state that you cannot provide an accurate forecast.

You have access to the following tools:
{tools}

Strict Rule: Do not guess or hallucinate weather patterns, economic trends or inflation rate. Use your tools to gather real-time or historical data for the target forecasting year before formulating your final answer.

STRICT EXECUTION RULES:
- You must search for BOTH past context and future forecasts. Do not jump to a final calculation until you have established the historical context of the years provided in the prompt.
- Do not make up weather patterns, economic trends, or inflation rates. If a tool query returns no results for a specific year, declare the data missing.

STRICT FORMATTING RULES:
You must follow the format below precisely.
- "Action:" must contain ONLY the exact name of the tool, chosen from [{tool_names}]. Do not write sentences, descriptions, or add code syntax here.
- "Action Input:" must contain the raw search query or argument for that tool.

Incorrect Example (Do NOT do this):
Action: Search for current and forecasted weather conditions in Brazil for 2022-2026
Action Input: Brazil weather 2026

Correct Example (Do this):
Action: Search
Action Input: weather conditions Brazil 2026

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)

Thought: I now know the final answer

Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt_assistant = PromptTemplate(template=template_task4, input_variables=["tools", "tool_names", "input", "agent_scratchpad"])

In [162]:
# Define an agent - react=reasoning and acton
agent_task4 = create_react_agent(llm_task4, tools_task4, prompt_assistant) # llm, tools, prompt - as input to the agent
agent_executor_task4 = AgentExecutor(agent=agent_task4, tools=tools_task4, verbose=True, handle_parsing_errors=True) # create agent executer for this agent; single agent can not be run

In [164]:
# Derive the answer to the question
agent_executor_task4.invoke({'input': "We export oranges from Brazil. In 2022 we exported 200t, in 2023 - 190t, in 2024 - 210t, in 2025 - 220t. Make an estimate of how many oranges we will be able to export in 2026, taking into account the weather conditions in Brazil and the global demand for oranges based on the economic situation."})



> Entering new AgentExecutor chain...
I need to gather historical data on weather conditions, economic factors, and export volumes for the years provided to establish a baseline for the forecast in 2026.

Action: Search
Action Input: Brazil weather conditions 2022-2025{'type': 'weather_result', 'temperature': '73', 'unit': 'Fahrenheit', 'precipitation': '0%', 'humidity': '72%', 'wind': '3 mph', 'location': 'Brazil', 'date': 'Sunday', 'weather': 'Sunny'}I have obtained the weather conditions for Brazil for the years 2022-2025. Now, I need to search for economic factors and export volumes for those years.

Action: Search
Action Input: Brazil GDP growth rate 2022-2025['GDP growth (annual %) - Brazil from The World Bank: Data.', 'GDP Annual Growth Rate in Brazil averaged 2.48 percent from 1991 until 2026, reaching an all time high of 12.40 percent in the second quarter of 2021 and a ...', "According to data released on 3 March 2026 by the Brazilian Institute of Geography and Statistics (

{'input': 'We export oranges from Brazil. In 2022 we exported 200t, in 2023 - 190t, in 2024 - 210t, in 2025 - 220t. Make an estimate of how many oranges we will be able to export in 2026, taking into account the weather conditions in Brazil and the global demand for oranges based on the economic situation.',
 'output': 'Based on the historical data of orange exports, weather conditions, and the forecasted economic situation in Brazil for 2026, it is estimated that the export volume of oranges in 2026 may be around 210-215t.'}